# Example: Stress-Testing the Minimum-Variance Portfolio

In this example, we take the minimum-variance allocation built in the previous notebook and stress-test it across 5,000 synthetic futures generated by the hybrid [JumpHMM](https://github.com/varnerlab/JumpHMM.jl) construction. We compare it head-to-head against an equal-weight 1/N benchmark, a buy-the-market benchmark, and a continuously compounded risk-free baseline on a full tail-risk scorecard — Value-at-Risk, Conditional VaR (Expected Shortfall), max drawdown — and on the **portfolio Net Present Value relative to the risk-free rate**, which tells us not whether the portfolio made money but whether it beat what the same dollars would have earned sitting in a risk-free asset. We then zoom into the worst 5% of paths to see what failure looks like.

> __Frictionless Assumptions:__
>
> Throughout this notebook (and the rest of Session 1) we make the standard introductory simplifications: zero transaction costs, infinitely divisible shares, perfect order fills at each day's stated price, no taxes, no slippage, no margin or borrowing limits beyond the long-only constraint, and instant execution. These assumptions are pedagogical scaffolding — like the frictionless plane in introductory physics — that let us see the structural ideas (Markowitz optimization, hybrid forward simulation, distributional scorecards) without the operational noise of real-world trading. Sessions 2 through 4 introduce the frictions where they matter.

> __Learning Objectives:__
>
> * __Tail-risk evaluation under hybrid futures:__ Run a large Monte Carlo of synthetic futures via the hybrid SIM construction and compute tail-focused metrics (Value-at-Risk, Conditional VaR, P5 wealth, P95 drawdown). The standard error on the CVaR estimate tells you whether your sample size is large enough.
> * __Portfolio NPV against the risk-free baseline:__ Score every risky portfolio against a continuously compounded 1-year risk-free baseline using the Net Present Value $\text{NPV}(r_f, T) = -W_{\mathcal{P}}(0) + W_{\mathcal{P}}(T)\,e^{-r_f T}$. Median NPV, CVaR of NPV, and the NPV-fail rate $P[\text{NPV}<0]$ replace the nominal "did it lose money?" question with the economically correct "did it beat risk-free?".
> * __Worst-case forensics and downstream baseline:__ Identify the worst 5% of paths, visualize their wealth trajectories alongside the median and the central corridor, and persist the scorecard plus raw per-path arrays (terminal wealth and NPV for all four portfolios) so the adaptive rebalancing engine in Session 2 has a clear baseline to beat.

For the formal definitions of VaR, CVaR, max drawdown, and portfolio NPV, see [the Tail-Risk Metrics and Portfolio NPV sections of the Session 1 lecture](eCornell-AI-Finance-S1-Lecture-StressTestingMinVariancePortfolios-May-2026.ipynb).

Let's get to work!
___

## Setup, Data, and Prerequisites
We begin by loading our packages and helper functions via the `Include.jl` file. This activates the local Julia environment and loads all dependencies.

This notebook depends on `code/src/data/minvar-allocation.jld2`, which is produced by the previous example. If you have not yet run the [BuildMinVariancePortfolio notebook](eCornell-AI-Finance-S1-Example-BuildMinVariancePortfolio-May-2026.ipynb), do that first — the load step in Task 1 will error otherwise.

In [1]:
# --- Load packages and helper functions ---
include("Include.jl"); # The Include.jl file activates the local Julia environment and imports all dependencies.

___
## Task 1: Load the Allocation and Run a 5,000-Path Stress Simulation
We load the calibrated minimum-variance allocation from the previous notebook, build the equal-weight benchmark, generate 5,000 forward paths from the hybrid SIM construction, and run buy-and-hold across four portfolios on the same set of futures: min-var, equal-weight, the market index, and a deterministic continuously compounded risk-free baseline.

> __What are we going to do?__
>
> Load `minvar-allocation.jld2`, derive the 1/N equal-weight vector, set the 1-year risk-free rate `r_f = 0.045`, pull the surrogate models and current prices, generate one large hybrid scenario via [the `generate_hybrid_scenario(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.generate_hybrid_scenario), and run buy-and-hold four times: once for the min-var weights, once for equal weights, once for the market index using [the `backtest_buyhold_market(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.backtest_buyhold_market), and once for a synthetic risk-free portfolio whose terminal wealth is the deterministic continuously compounded value $B_0 \cdot e^{r_f \cdot T}$ on every path. The same `scenario` is reused for all four portfolios so every comparison is on the same set of synthetic futures.

The code below loads the allocation file and the surrogate models, then declares the stress-test parameters.

In [2]:
B₀, r_f, allocation_weights, my_tickers, eq_weights, sim_estimates, σ_m = let

    # --- Step 1: Load the calibrated min-var allocation from the previous notebook ---
    minvar = load_results(joinpath(_PATH_TO_DATA, "minvar-allocation.jld2"));
    tickers = minvar["my_tickers"]::Vector{String};
    weights = Float64.(minvar["allocation_weights"]);
    estimates = minvar["sim_estimates"];
    σm = Float64(minvar["sigma_market"]);

    # --- Step 2: Build the 1/N equal-weight benchmark ---
    N = length(tickers);
    eqw = fill(1.0 / N, N);

    # --- Step 3: Risk-free baseline rate (1-year STRIPS) ---
    # 4.5% annualized — change this to whatever the prevailing 1-year zero-coupon
    # Treasury yield is on the day you run the notebook.
    rf = 0.045;

    # --- Step 4: Display loaded configuration ---
    println("Loaded minimum-variance allocation:")
    println("  $(N) tickers: $(tickers)")
    println("  σ_m (annualized growth rate): $(round(σm, digits=4))")
    println("  Min-var weights:    $(round.(weights .* 100, digits=2))%")
    println("  Equal-weight bench: $(round.(eqw .* 100, digits=2))%")
    println("  Risk-free baseline: $(round(rf*100, digits=2))% / year (1-year STRIPS)")

    10_000.0, rf, weights, tickers, eqw, estimates, σm
end;

Loaded minimum-variance allocation:
  10 tickers: ["AAPL", "MSFT", "NVDA", "JNJ", "JPM", "PG", "XOM", "BA", "GS", "AMD"]
  σ_m (annualized growth rate): 2.145
  Min-var weights:    [9.91, 14.1, 4.91, 32.62, 0.0, 38.46, -0.0, -0.0, -0.0, -0.0]%
  Equal-weight bench: [10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0]%
  Risk-free baseline: 4.5% / year (1-year STRIPS)


With the allocation in hand we generate a single large hybrid forward scenario with **5,000 paths × 252 trading days × N tickers**, then run [the `backtest_buyhold(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.backtest_buyhold) for the min-var and equal-weight portfolios on `scenario.price_paths`, and [the `backtest_buyhold_market(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.backtest_buyhold_market) for the market index on `scenario.market_paths`. All three results have the same `MyBacktestResult` shape so the scorecard code can iterate over them uniformly.

In [ ]:
scenario, result_mv, result_eq, result_mkt, result_rf, n_paths, n_steps = let

    # --- Step 1: Stress simulation parameters ---
    n_paths = 5_000;
    n_steps = 252;
    Δt      = 1.0 / 252.0;
    seed    = 2026;

    # --- Step 2: Load the surrogate models and the start-price snapshot ---
    market_model = MyMarketSurrogateModel();
    portfolio    = MyPortfolioSurrogateModel();
    calib        = MySIMCalibration();

    snap = MyCurrentPrices();
    snap_lookup = Dict(snap["tickers"] .=> snap["prices"]);
    start_prices = Dict(t => snap_lookup[t] for t ∈ my_tickers);

    # --- Step 3: Generate ONE 5,000-path hybrid scenario ---
    println("Generating $(n_paths)-path hybrid scenario ($(n_steps) days)...")
    scen = generate_hybrid_scenario(market_model, portfolio, calib, my_tickers;
        n_paths = n_paths,
        n_steps = n_steps,
        Δt      = Δt,
        start_prices = start_prices,
        label   = "Stress-Test ($(n_paths)p × $(n_steps)d)",
        seed    = seed);
    println("  scenario.price_paths shape:  $(size(scen.price_paths))")
    println("  scenario.market_paths shape: $(size(scen.market_paths))")

    # --- Step 4: Run buy-and-hold for the three risky portfolios ---
    r_mv  = backtest_buyhold(scen, my_tickers; B₀=B₀, offset=1, weights=allocation_weights);
    r_eq  = backtest_buyhold(scen, my_tickers; B₀=B₀, offset=1, weights=eq_weights);
    r_mkt = backtest_buyhold_market(scen; B₀=B₀, offset=1);

    # --- Step 5: Synthesize the risk-free baseline (continuous compounding) ---
    # Every path ends at the same deterministic terminal wealth: B₀·exp(r_f · T).
    # This is the same continuous compounding convention used everywhere else
    # in Session 1 (CCGR units). It is also the value we discount BACK to today
    # via NPV(r_f, T) — so the risk-free portfolio has NPV ≡ 0 by construction.
    # Drawdown is identically zero, Sharpe convention 0 (zero excess vol), nominal fail rate 0.
    T_years          = n_steps / 252;
    rf_terminal      = B₀ * exp(r_f * T_years);
    r_rf = MyBacktestResult();
    r_rf.scenario_label = scen.label;
    r_rf.strategy_label = "Risk-Free Buy-and-Hold (continuous compounding)";
    r_rf.final_wealth   = fill(rf_terminal, n_paths);
    r_rf.max_drawdowns  = zeros(n_paths);
    r_rf.sharpe_ratios  = zeros(n_paths);

    println("\nBuy-and-hold complete for all four portfolios:")
    println("  Min-var:      median terminal wealth = \$$(round(median(r_mv.final_wealth), digits=0))")
    println("  Equal-weight: median terminal wealth = \$$(round(median(r_eq.final_wealth), digits=0))")
    println("  Market:       median terminal wealth = \$$(round(median(r_mkt.final_wealth), digits=0))")
    println("  Risk-free:    deterministic terminal wealth = \$$(round(rf_terminal, digits=0))  (B₀·exp(r_f·T))")

    scen, r_mv, r_eq, r_mkt, r_rf, n_paths, n_steps
end;

___
## Task 2: Tail-Risk Scorecard with Portfolio NPV
We compute the full tail-focused scorecard for all four portfolios — median terminal wealth, P5 (Value-at-Risk), Conditional VaR (Expected Shortfall) at the 5% level with sampling standard error, max-drawdown percentiles, median Sharpe — and then add three **portfolio NPV** columns that score every risky portfolio against the continuously compounded risk-free baseline: median NPV in dollars, the CVaR of NPV (worst-tail discounted excess loss), and the NPV-fail rate $P[\text{NPV} < 0]$.

For the formal definitions of each metric (VaR, CVaR + sampling SE, max drawdown, and the portfolio NPV equation $\text{NPV}(r_f, T) = -W_{\mathcal{P}}(0) + W_{\mathcal{P}}(T)\,e^{-r_f T}$), see [the Tail-Risk Metrics and Portfolio NPV sections of the Session 1 lecture](eCornell-AI-Finance-S1-Lecture-StressTestingMinVariancePortfolios-May-2026.ipynb).

> __What are we going to do?__
>
> For each of the four `MyBacktestResult` objects, compute the per-path NPV via `npv = result.final_wealth .* exp(-r_f * T_years) .- B₀`, then assemble the wealth-based and NPV-based metrics into a single comparison `DataFrame` and display it via the canonical `pretty_table` call. The risk-free row will collapse to a deterministic point: median NPV exactly zero, NPV-fail rate exactly zero — by construction, since the risk-free portfolio _is_ the discounted baseline.

The code below builds the scorecard and stores it in `scorecard::DataFrame`.

In [ ]:
scorecard = let

    # --- Discount factor for the NPV calculation: e^(-r_f · T) ---
    T_years      = n_steps / 252;
    discount     = exp(-r_f * T_years);

    # --- Helper: compute the NPV per-path array for a result ---
    npv_array(r::MyBacktestResult) = r.final_wealth .* discount .- B₀;

    # --- Helper: row of metrics for one MyBacktestResult ---
    function tail_metrics(name::String, r::MyBacktestResult, B0::Float64)
        wealth = r.final_wealth;
        npv    = npv_array(r);
        n      = length(wealth);
        n_tail = max(1, floor(Int, 0.05 * n));

        # wealth-side tail
        sorted_wealth = sort(wealth);
        wealth_tail   = sorted_wealth[1:n_tail];
        cvar5         = mean(wealth_tail);
        cvar5_se      = (length(wealth_tail) > 1) ? std(wealth_tail) / sqrt(n_tail) : 0.0;

        # NPV-side tail (worst NPVs = most-negative)
        sorted_npv    = sort(npv);
        npv_tail      = sorted_npv[1:n_tail];
        cvar5_npv     = mean(npv_tail);

        return (
            Portfolio    = name,
            Median_W     = round(median(wealth),         digits=0),
            VaR_5        = round(quantile(wealth, 0.05), digits=0),
            CVaR_5       = round(cvar5,                  digits=0),
            CVaR_5_SE    = round(cvar5_se,               digits=0),
            Med_DD_pct   = round(median(r.max_drawdowns) * 100,         digits=2),
            P95_DD_pct   = round(quantile(r.max_drawdowns, 0.95) * 100, digits=2),
            Med_Sharpe   = round(median(r.sharpe_ratios), digits=3),
            Med_NPV      = round(median(npv),            digits=0),
            NPV_scaled   = round(median(npv) / B0 * 100, digits=2),  # %
            CVaR5_NPV    = round(cvar5_npv,              digits=0),
            P_NPV_neg    = round(mean(npv .< 0) * 100,   digits=1),  # %
        );
    end

    # --- Build the scorecard rows ---
    rows = [
        tail_metrics("Min-Var",         result_mv,  B₀),
        tail_metrics("Equal-Weight",    result_eq,  B₀),
        tail_metrics("Market (SPY)",    result_mkt, B₀),
        tail_metrics("Risk-Free $(round(r_f*100, digits=1))%", result_rf, B₀),
    ];
    df = DataFrame(rows);

    # --- Display via canonical pretty_table kwargs ---
    println("Tail-risk scorecard over $(n_paths) paths, $(n_steps)-day horizon (T = $(round(T_years, digits=3)) yr):")
    println("Discount factor e^(-r_f·T) = $(round(discount, digits=5))")
    println()
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    # --- Print convergence note (skip risk-free which is deterministic) ---
    n_tail = floor(Int, 0.05 * n_paths);
    println()
    println("CVaR sample size: $(n_tail) paths in the 5% tail.")
    println("CVaR standard errors above are analytical (std(tail) / √n_tail).")
    println("Standard-error / point-estimate ratios:")
    for row ∈ eachrow(df)
        if row.CVaR_5 > 0 && row.CVaR_5_SE > 0
            ratio = row.CVaR_5_SE / abs(row.CVaR_5);
            println("  $(rpad(row.Portfolio, 20)) SE / |CVaR| = $(round(ratio*100, digits=2))%")
        else
            println("  $(rpad(row.Portfolio, 20)) deterministic — SE = 0 by construction")
        end
    end
    println("Rule of thumb: if any non-deterministic SE/|CVaR| ratio exceeds ~1%, increase n_paths.")
    println()
    println("NPV interpretation:")
    println("  Med_NPV    : median (\$) excess wealth in today's dollars vs the risk-free baseline")
    println("  NPV_scaled : Med_NPV expressed as % of initial wealth B₀")
    println("  CVaR5_NPV  : mean (\$) NPV across the worst 5% of paths (most negative)")
    println("  P_NPV_neg  : % of paths on which the portfolio underperformed risk-free")
    println("  Risk-free row is identically zero on all NPV columns by construction.")

    df
end;

The histograms below show three views of the same Monte Carlo ensemble: (1) terminal wealth scaled by initial wealth, (2) max drawdown, and (3) **portfolio NPV in dollars**. On the wealth panel, the dashed black line at `1.0` is break-even and the green line at $e^{r_f T}$ is the risk-free baseline. On the NPV panel, **NPV $= 0$ is the risk-free baseline by construction** — paths to the left of zero underperformed risk-free, paths to the right beat it. Look for separation in the left tail of the wealth panel, the right tail of the drawdown panel, and the left tail of the NPV panel — that is where the tail risk lives.

In [ ]:
let
    T_years     = n_steps / 252;
    discount    = exp(-r_f * T_years);
    rf_multiple = result_rf.final_wealth[1] / B₀;  # exp(r_f · T) for the horizon

    # Per-path NPV arrays for the three risky portfolios
    npv_mv  = result_mv.final_wealth  .* discount .- B₀;
    npv_eq  = result_eq.final_wealth  .* discount .- B₀;
    npv_mkt = result_mkt.final_wealth .* discount .- B₀;

    # --- Panel 1: terminal wealth overlay (scaled by B₀) ---
    p1 = histogram(result_mv.final_wealth ./ B₀, bins=60, alpha=0.55, color=:red,
        label="Min-Var", xlabel="Terminal Wealth / Initial Wealth", ylabel="Count",
        fontsize=18);
    histogram!(p1, result_eq.final_wealth ./ B₀, bins=60, alpha=0.55, color=:gray20, label="Equal-Weight");
    histogram!(p1, result_mkt.final_wealth ./ B₀, bins=60, alpha=0.55, color=:navy, label="Market");
    vline!(p1, [1.0], lw=2, ls=:dash, c=:black, label="Initial (1.0)");
    vline!(p1, [rf_multiple], lw=3, c=:seagreen, label="Risk-Free ($(round(rf_multiple, digits=3)))");
    plot!(p1, bg="gray95", background_color_outside="white",
        framestyle=:box, fg_legend=:transparent, legend=:topright);

    # --- Panel 2: max drawdown overlay ---
    p2 = histogram(result_mv.max_drawdowns .* 100, bins=60, alpha=0.55, color=:red,
        label="Min-Var", xlabel="Max Drawdown (%)", ylabel="Count",
        fontsize=18);
    histogram!(p2, result_eq.max_drawdowns .* 100, bins=60, alpha=0.55, color=:gray20, label="Equal-Weight");
    histogram!(p2, result_mkt.max_drawdowns .* 100, bins=60, alpha=0.55, color=:navy, label="Market");
    vline!(p2, [0.0], lw=3, c=:seagreen, label="Risk-Free (0%)");
    plot!(p2, bg="gray95", background_color_outside="white",
        framestyle=:box, fg_legend=:transparent, legend=:topright);

    # --- Panel 3: portfolio NPV overlay (dollars) ---
    p3 = histogram(npv_mv, bins=60, alpha=0.55, color=:red,
        label="Min-Var", xlabel="Portfolio NPV (\$)", ylabel="Count",
        fontsize=18);
    histogram!(p3, npv_eq,  bins=60, alpha=0.55, color=:gray20, label="Equal-Weight");
    histogram!(p3, npv_mkt, bins=60, alpha=0.55, color=:navy,   label="Market");
    vline!(p3, [0.0], lw=3, c=:seagreen, label="Risk-Free (NPV=0)");
    plot!(p3, bg="gray95", background_color_outside="white",
        framestyle=:box, fg_legend=:transparent, legend=:topright);

    plot(p1, p2, p3, layout=(1, 3), size=(1800, 460), margin=5Plots.mm)
end

___
## Task 3: Worst-Case Forensics and Hand-Off Baseline
We zoom into the worst 5% of paths for the minimum-variance portfolio (the same 250 paths underlying its CVaR), plot their wealth trajectories alongside the median and the central P5–P95 corridor, print a one-paragraph stress narrative, and save the full scorecard plus raw arrays to disk as the baseline that the adaptive rebalancing engine in Session 2 must beat.

> __What are we going to do?__
>
> Identify the indices of the worst 5% of paths sorted by min-var terminal wealth, recompute their wealth trajectories from `scenario.price_paths`, plot them in red against the all-paths median (thick black) and a P5–P95 gray band, then save `stress-test-baseline.jld2` with the scorecard, the raw final-wealth / drawdown / Sharpe arrays for all three portfolios, and the seed / `n_paths` / `n_steps` for reproducibility.

The code below identifies the worst-case paths and renders the trajectory plot.

In [ ]:
let
    # --- Step 1: Identify the worst 5% of paths for the min-var portfolio ---
    n_tail = max(1, floor(Int, 0.05 * n_paths));
    worst_idx = partialsortperm(result_mv.final_wealth, 1:n_tail);
    IW = B₀;  # initial wealth — scale wealth trajectories so 1.0 = break-even
    rf_multiple = result_rf.final_wealth[1] / B₀;  # 1 + r_f over the horizon

    # --- Step 2: Recompute wealth trajectories for ALL paths (min-var) ---
    K = length(my_tickers);
    wealth_all = zeros(n_steps, n_paths);
    for p ∈ 1:n_paths
        shares = [(B₀ * allocation_weights[k]) / scenario.price_paths[p, 1, k] for k ∈ 1:K];
        for t ∈ 1:n_steps
            wealth_all[t, p] = sum(shares[k] * scenario.price_paths[p, t, k] for k ∈ 1:K);
        end
    end

    # --- Step 3: Median + P5–P95 corridor + worst-tail paths ---
    median_traj = [median(wealth_all[t, :]) for t ∈ 1:n_steps];
    p05_traj    = [quantile(wealth_all[t, :], 0.05) for t ∈ 1:n_steps];
    p95_traj    = [quantile(wealth_all[t, :], 0.95) for t ∈ 1:n_steps];

    # --- Step 4: Plot the corridor + worst-tail in red, scaled by IW ---
    p = plot(1:n_steps, p95_traj ./ IW,
        fillrange = p05_traj ./ IW, fillalpha = 0.20, fillcolor = :gray60,
        lw = 0, label = "P5–P95 corridor",
        xlabel = "Trading Day",
        ylabel = "Wealth Multiple (W_t / B₀)",
        fontsize = 18,
        size = (850, 480));
    # the worst 5% of paths in red, lightly drawn
    for idx ∈ worst_idx
        plot!(p, 1:n_steps, wealth_all[:, idx] ./ IW,
            lw = 1, c = :red, alpha = 0.25, label = "");
    end
    plot!(p, 1:n_steps, median_traj ./ IW, lw = 4, c = :gray20, label = "Median (all paths)");
    hline!(p, [1.0], lw = 2, ls = :dash, c = :black, label = "Initial (1.0)");
    hline!(p, [rf_multiple], lw = 2, ls = :dash, c = :seagreen,
        label = "Risk-Free terminal ($(round(rf_multiple, digits=3)))");
    plot!(p, bg = "gray95",
        background_color_outside = "white",
        framestyle = :box,
        fg_legend = :transparent,
        legend = :topleft);
    title!(p, "Min-Var: worst $(n_tail) of $(n_paths) paths (red) vs corridor")
    display(p)
end

The narrative below summarizes how each portfolio behaves in the worst 5% of paths in **NPV terms** — the discounted excess (or deficit) in today's dollars relative to the risk-free baseline. Reporting losses against the risk-free baseline rather than against initial wealth is the economically meaningful framing: a portfolio that holds its nominal value but underperforms a 4.5%/yr risk-free asset has still cost the investor real opportunity.

In [ ]:
let
    n_tail   = max(1, floor(Int, 0.05 * n_paths));
    T_years  = n_steps / 252;
    discount = exp(-r_f * T_years);

    # Per-path NPV in dollars: NPV(r_f, T) = -B₀ + W_T · e^(-r_f T)
    npv_of(r) = r.final_wealth .* discount .- B₀;

    function npv_tail_stats(npv::Vector{Float64}, B0::Float64, ntail::Int)
        sorted = sort(npv);
        worst  = sorted[1:ntail];
        return (
            median_npv      = median(npv),
            cvar5_npv       = mean(worst),
            worst_npv       = minimum(worst),
            p_npv_neg       = mean(npv .< 0) * 100,
            median_loss_pct = -median(worst) / B0 * 100,  # losses vs risk-free as % of B₀
            worst_loss_pct  = -minimum(worst) / B0 * 100,
        );
    end

    mv  = npv_tail_stats(npv_of(result_mv),  B₀, n_tail);
    eq  = npv_tail_stats(npv_of(result_eq),  B₀, n_tail);
    mkt = npv_tail_stats(npv_of(result_mkt), B₀, n_tail);
    # Risk-free is deterministic — every path has NPV ≡ 0 by construction.
    rf_npv = 0.0;

    println("="^80)
    println("STRESS NARRATIVE — worst 5% of $(n_paths) paths ($(n_tail) tail paths), measured in NPV terms")
    println("="^80)
    println()
    println("Median portfolio NPV across ALL $(n_paths) paths (vs risk-free baseline, in today's dollars):")
    println("  Min-Var:      \$$(round(mv.median_npv,  digits=0))   ($(round(mv.p_npv_neg,  digits=1))% of paths underperformed risk-free)")
    println("  Equal-Weight: \$$(round(eq.median_npv,  digits=0))   ($(round(eq.p_npv_neg,  digits=1))% of paths underperformed risk-free)")
    println("  Market:       \$$(round(mkt.median_npv, digits=0))   ($(round(mkt.p_npv_neg, digits=1))% of paths underperformed risk-free)")
    println("  Risk-Free:    \$$(round(rf_npv, digits=0))   (NPV ≡ 0 by construction — the baseline)")
    println()
    println("Worst-tail (5%) NPV — average of the 250 most-negative paths:")
    println("  Min-Var:      CVaR_5 NPV = \$$(round(mv.cvar5_npv,  digits=0))   (worst single path: \$$(round(mv.worst_npv,  digits=0)))")
    println("  Equal-Weight: CVaR_5 NPV = \$$(round(eq.cvar5_npv,  digits=0))   (worst single path: \$$(round(eq.worst_npv,  digits=0)))")
    println("  Market:       CVaR_5 NPV = \$$(round(mkt.cvar5_npv, digits=0))   (worst single path: \$$(round(mkt.worst_npv, digits=0)))")
    println()
    println("Equivalent percentage loss vs the risk-free baseline in the worst 5% of paths:")
    println("  Min-Var:      median loss vs risk-free = $(round(mv.median_loss_pct,  digits=1))% of B₀")
    println("  Equal-Weight: median loss vs risk-free = $(round(eq.median_loss_pct,  digits=1))% of B₀")
    println("  Market:       median loss vs risk-free = $(round(mkt.median_loss_pct, digits=1))% of B₀")
    println()
    advantage_eq  = eq.median_loss_pct  - mv.median_loss_pct;
    advantage_mkt = mkt.median_loss_pct - mv.median_loss_pct;
    println("Min-Var advantage in the worst 5% (positive = min-var loses LESS vs risk-free than benchmark):")
    println("  vs Equal-Weight: $(round(advantage_eq,  digits=1)) percentage points")
    println("  vs Market (SPY): $(round(advantage_mkt, digits=1)) percentage points")
    println()
    println("Interpretation: positive median NPV means the portfolio beat risk-free in the typical")
    println("future; the CVaR_5 NPV is the average dollar deficit vs risk-free in the bad tail; the")
    println("NPV-fail rate is the probability the portfolio underperformed cash earning r_f.")
    println("="^80)
end

Finally we save the stress-test baseline to disk for downstream sessions. The next notebook (Session 2's adaptive rebalancing engine) will load this file and try to beat the buy-and-hold tail metrics under the same set of futures.

The code below saves the handoff file using [the `save_results(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.save_results).

In [ ]:
let
    save_path = joinpath(_PATH_TO_DATA, "stress-test-baseline.jld2");
    T_years   = n_steps / 252;
    discount  = exp(-r_f * T_years);
    npv_of(r) = r.final_wealth .* discount .- B₀;

    save_results(save_path, Dict(
        "my_tickers"         => my_tickers,
        "allocation_weights" => allocation_weights,
        "eq_weights"         => eq_weights,
        # raw per-path arrays for all four portfolios
        "mv_final_wealth"    => result_mv.final_wealth,
        "mv_max_drawdowns"   => result_mv.max_drawdowns,
        "mv_sharpe_ratios"   => result_mv.sharpe_ratios,
        "mv_npv"             => npv_of(result_mv),
        "eq_final_wealth"    => result_eq.final_wealth,
        "eq_max_drawdowns"   => result_eq.max_drawdowns,
        "eq_sharpe_ratios"   => result_eq.sharpe_ratios,
        "eq_npv"             => npv_of(result_eq),
        "mkt_final_wealth"   => result_mkt.final_wealth,
        "mkt_max_drawdowns"  => result_mkt.max_drawdowns,
        "mkt_sharpe_ratios"  => result_mkt.sharpe_ratios,
        "mkt_npv"            => npv_of(result_mkt),
        "rf_final_wealth"    => result_rf.final_wealth,
        "rf_max_drawdowns"   => result_rf.max_drawdowns,
        "rf_sharpe_ratios"   => result_rf.sharpe_ratios,
        "rf_npv"             => npv_of(result_rf),  # ≡ 0 by construction
        # scorecard summary as a Dict (JLD2 round-trips this cleanly)
        "scorecard"          => Dict(string(c) => scorecard[!, c] for c ∈ names(scorecard)),
        # reproducibility metadata
        "B₀"                 => B₀,
        "r_f"                => r_f,
        "discount_factor"    => discount,   # e^(-r_f · T)
        "T_years"            => T_years,
        "n_paths"            => n_paths,
        "n_steps"            => n_steps,
        "seed"               => 2026,
    ));

    println("Saved stress-test baseline to: $(save_path)")
    println("  Four portfolios × $(n_paths) paths × $(n_steps) days")
    println("  Per-path arrays: terminal wealth, max drawdown, Sharpe, NPV(r_f, T)")
    println("  Discount factor used: e^(-r_f·T) = $(round(discount, digits=5))")
    println("  Scorecard rows: $(nrow(scorecard))")
    println("  Hand-off ready for Session 2 adaptive rebalancing engine.")
end;

___
## Summary
This example stress-tested the calibrated minimum-variance allocation across 5,000 hybrid synthetic futures, compared it head-to-head against an equal-weight benchmark, a buy-the-market benchmark, and a continuously compounded risk-free baseline using both wealth-based tail-risk metrics and the **portfolio NPV** against the risk-free rate, zoomed into the worst 5% of paths to see what failure looks like in NPV terms, and saved the resulting scorecard plus per-path NPV arrays as the baseline that the adaptive rebalancing engine in Session 2 must beat.

> __Key Takeaways:__
>
> * __Tail risk is a distribution, not a number:__ Median terminal wealth is the easy summary, but Conditional VaR (Expected Shortfall) at the 5% level captures the *average* loss in the bad tail and is the more informative measure of downside risk. The standard error on the CVaR estimate tells you whether your sample size is large enough to trust the number.
> * __NPV is the right way to score against a benchmark:__ Reporting a portfolio's "fail rate" against initial wealth $B_0$ asks the wrong question — a portfolio that broke even nominally still failed if a risk-free asset would have earned the investor 4.5%. The NPV-fail rate $P[\text{NPV} < 0]$ measures failure against the risk-free baseline, the median NPV measures the typical excess in today's dollars, and the CVaR of NPV measures the worst-tail discounted opportunity cost — all on the same `e^{-r_f T}`-discounted scale.
> * __The baseline scorecard is the bar Session 2 must clear:__ The min-var buy-and-hold outcome — wealth-based metrics, NPV-based metrics, and the raw per-path arrays — is persisted to disk so the adaptive rebalancing engine can be evaluated against the same set of futures. Session 2 wins only if rebalancing tightens the NPV tail or lifts the median NPV without inflating the drawdown.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.

___